<a href="https://colab.research.google.com/github/efitznm/Data-Science-Portfolio-C22/blob/main/Modules/Module_3_SQL/Projects/Instructions/Project.3-SQL/SQL_1_Chinook_Phase_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



### Phase 3. Answer Guided Questions



Once you know the syntax and understand the data structure, you will put your skills to work solving specific business problems. Below are sets of explicit questions to answer using SQL. This phase mimics standard reporting: you are handed a specific prompt, and your job is to translate that prompt into an accurate, data-driven answer.

Here are some practice questions:
- https://github.com/LucasMcL/15-sql_queries_02-chinook/blob/master/README.md
- [Using the R language]( https://rpubs.com/enext777/636199 )
- [Search Google]( https://www.google.com/search?q=chinook+database+questions )
- Ask AI to generate some questions for you.  Be sure to mention the Chinook data set.




## Why This Matters: The Path to Your Capstone


Learning to ask your own questions is the exact skill you will rely on for your final capstone project. In your capstone, you will choose a dataset from your own field of interest. No one will hand you a list of prompts or a guide telling you what to look for.

Use this project as your sandbox. Mastering the ability to interrogate a database now will give you the framework you need to discover meaningful, independent insights in your capstone later.



## Chinook data set

See the lecture on SQLite3 using the Chinook data set to set up the software, database, and tables, as well as for the links to ancillary information about the data set.


In [1]:
# Import libraries
import sqlite3 as db
import pandas as pd
import contextlib
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Install the sqlite package for Ubuntu
%%capture
%%bash
apt-get update
apt-get install -y sqlite3


In [3]:
# Download chinook.zip silently (-s) if it does not already exist in the current directory
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
# List (-l) the contents of the downloaded zip archive without extracting it yet
unzip -l chinook.zip


Archive:  chinook.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   884736  2015-11-29 10:53   chinook.db
---------                     -------
   884736                     1 file


In [4]:
# Force-remove any existing chinook.db file to ensure a clean extraction
!rm -f chinook.db


In [5]:
# Unzip chinook.zip (-u updates/creates files) to extract chinook.db into the working directory
!unzip -u chinook.zip


Archive:  chinook.zip
  inflating: chinook.db              


In [6]:
# List all files in the current directory with detailed permissions, sizes, and timestamps
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 16 20:34 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


In [7]:
# Confirm specifically that the .db file exists and check its file size
!ls -l *.db

-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db


In [8]:
# Confirm the database has tables
%%script sqlite3 --column --header chinook.db
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;

table_name    
--------------
albums        
artists       
customers     
employees     
genres        
invoice_items 
invoices      
media_types   
playlist_track
playlists     
tracks        


## Use SQL queries to pull specific information

Do NOT pull all the data and then filter using DataFrame methods.  The initial purpose is to get familiar with SQL.




### 1. Customers not in the US

In [9]:
%%script sqlite3 --column --header chinook.db
SELECT FirstName || ' ' || LastName AS FullName, CustomerId, Country
FROM customers
WHERE Country <> 'USA';

FullName               CustomerId  Country       
---------------------  ----------  --------------
Luís Gonçalves         1           Brazil        
Leonie Köhler          2           Germany       
François Tremblay      3           Canada        
Bjørn Hansen           4           Norway        
František Wichterlová  5           Czech Republic
Helena Holý            6           Czech Republic
Astrid Gruber          7           Austria       
Daan Peeters           8           Belgium       
Kara Nielsen           9           Denmark       
Eduardo Martins        10          Brazil        
Alexandre Rocha        11          Brazil        
Roberto Almeida        12          Brazil        
Fernanda Ramos         13          Brazil        
Mark Philips           14          Canada        
Jennifer Peterson      15          Canada        
Robert Brown           29          Canada        
Edward Francis         30          Canada        
Martha Silk            31          Canada        


### 2. Customers from Brazil

In [10]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM customers
WHERE Country = 'Brazil';

CustomerId  FirstName  LastName   Company                                           Address                          City                 State  Country  PostalCode  Phone               Fax                 Email                          SupportRepId
----------  ---------  ---------  ------------------------------------------------  -------------------------------  -------------------  -----  -------  ----------  ------------------  ------------------  -----------------------------  ------------
1           Luís       Gonçalves  Embraer - Empresa Brasileira de Aeronáutica S.A.  Av. Brigadeiro Faria Lima, 2170  São José dos Campos  SP     Brazil   12227-000   +55 (12) 3923-5555  +55 (12) 3923-5566  luisg@embraer.com.br           3           
10          Eduardo    Martins    Woodstock Discos                                  Rua Dr. Falcão Filho, 155        São Paulo            SP     Brazil   01007-010   +55 (11) 3033-5446  +55 (11) 3033-4564  eduardo@woodstock.com.br       4           


### 3. Invoices of Brazilian customers (name, invoice ID, date, billing country)

In [11]:
%%script sqlite3 --column --header chinook.db
SELECT c.FirstName || ' ' || c.LastName AS FullName,
       i.InvoiceId, i.InvoiceDate, i.BillingCountry
FROM customers AS c
JOIN invoices AS i ON i.CustomerId = c.CustomerId
WHERE c.Country = 'Brazil';

FullName         InvoiceId  InvoiceDate          BillingCountry
---------------  ---------  -------------------  --------------
Luís Gonçalves   98         2010-03-11 00:00:00  Brazil        
Luís Gonçalves   121        2010-06-13 00:00:00  Brazil        
Luís Gonçalves   143        2010-09-15 00:00:00  Brazil        
Luís Gonçalves   195        2011-05-06 00:00:00  Brazil        
Luís Gonçalves   316        2012-10-27 00:00:00  Brazil        
Luís Gonçalves   327        2012-12-07 00:00:00  Brazil        
Luís Gonçalves   382        2013-08-07 00:00:00  Brazil        
Eduardo Martins  25         2009-04-09 00:00:00  Brazil        
Eduardo Martins  154        2010-11-14 00:00:00  Brazil        
Eduardo Martins  177        2011-02-16 00:00:00  Brazil        
Eduardo Martins  199        2011-05-21 00:00:00  Brazil        
Eduardo Martins  251        2012-01-09 00:00:00  Brazil        
Eduardo Martins  372        2013-07-02 00:00:00  Brazil        
Eduardo Martins  383        2013-08-12 0

### 4. Employees who are sales agents

In [12]:
%%script sqlite3 --column --header chinook.db
SELECT *
FROM employees
WHERE Title = 'Sales Support Agent';

EmployeeId  LastName  FirstName  Title                ReportsTo  BirthDate            HireDate             Address           City     State  Country  PostalCode  Phone              Fax                Email                   
----------  --------  ---------  -------------------  ---------  -------------------  -------------------  ----------------  -------  -----  -------  ----------  -----------------  -----------------  ------------------------
3           Peacock   Jane       Sales Support Agent  2          1973-08-29 00:00:00  2002-04-01 00:00:00  1111 6 Ave SW     Calgary  AB     Canada   T2P 5M5     +1 (403) 262-3443  +1 (403) 262-6712  jane@chinookcorp.com    
4           Park      Margaret   Sales Support Agent  2          1947-09-19 00:00:00  2003-05-03 00:00:00  683 10 Street SW  Calgary  AB     Canada   T2P 5G3     +1 (403) 263-4423  +1 (403) 263-4289  margaret@chinookcorp.com
5           Johnson   Steve      Sales Support Agent  2          1965-03-03 00:00:00  2003-10-17 00:

### 5. Unique billing countries

In [13]:
%%script sqlite3 --column --header chinook.db
SELECT DISTINCT BillingCountry
FROM invoices
ORDER BY BillingCountry;

BillingCountry
--------------
Argentina     
Australia     
Austria       
Belgium       
Brazil        
Canada        
Chile         
Czech Republic
Denmark       
Finland       
France        
Germany       
Hungary       
India         
Ireland       
Italy         
Netherlands   
Norway        
Poland        
Portugal      
Spain         
Sweden        
USA           
United Kingdom


### 6. Invoices of Brazilian customers (all invoice columns)

In [14]:
%%script sqlite3 --column --header chinook.db
SELECT i.*
FROM invoices AS i
JOIN customers AS c ON c.CustomerId = i.CustomerId
WHERE c.Country = 'Brazil';

InvoiceId  CustomerId  InvoiceDate          BillingAddress                   BillingCity          BillingState  BillingCountry  BillingPostalCode  Total
---------  ----------  -------------------  -------------------------------  -------------------  ------------  --------------  -----------------  -----
98         1           2010-03-11 00:00:00  Av. Brigadeiro Faria Lima, 2170  São José dos Campos  SP            Brazil          12227-000          3.98 
121        1           2010-06-13 00:00:00  Av. Brigadeiro Faria Lima, 2170  São José dos Campos  SP            Brazil          12227-000          3.96 
143        1           2010-09-15 00:00:00  Av. Brigadeiro Faria Lima, 2170  São José dos Campos  SP            Brazil          12227-000          5.94 
195        1           2011-05-06 00:00:00  Av. Brigadeiro Faria Lima, 2170  São José dos Campos  SP            Brazil          12227-000          0.99 
316        1           2012-10-27 00:00:00  Av. Brigadeiro Faria Lima, 2170  São J

### 7. Invoices for each sales agent

In [15]:
%%script sqlite3 --column --header chinook.db
SELECT e.FirstName || ' ' || e.LastName AS SalesAgent, i.*
FROM invoices AS i
JOIN customers AS c ON c.CustomerId = i.CustomerId
JOIN employees AS e ON e.EmployeeId = c.SupportRepId
ORDER BY SalesAgent, i.InvoiceId;

SalesAgent     InvoiceId  CustomerId  InvoiceDate          BillingAddress                            BillingCity          BillingState  BillingCountry  BillingPostalCode  Total
-------------  ---------  ----------  -------------------  ----------------------------------------  -------------------  ------------  --------------  -----------------  -----
Jane Peacock   6          37          2009-01-19 00:00:00  Berger Straße 10                          Frankfurt                          Germany         60316              0.99 
Jane Peacock   7          38          2009-02-01 00:00:00  Barbarossastraße 19                       Berlin                             Germany         10779              1.98 
Jane Peacock   9          42          2009-02-02 00:00:00  9, Place Louis Barthou                    Bordeaux                           France          33000              3.96 
Jane Peacock   10         46          2009-02-03 00:00:00  3 Chatham Street                          Dublin        

### 8. Invoice total, customer, country and sales agent

In [16]:
%%script sqlite3 --column --header chinook.db
SELECT i.Total AS InvoiceTotal,
       c.FirstName || ' ' || c.LastName AS CustomerName,
       c.Country,
       e.FirstName || ' ' || e.LastName AS SalesAgent
FROM invoices AS i
JOIN customers AS c ON c.CustomerId = i.CustomerId
LEFT JOIN employees AS e ON e.EmployeeId = c.SupportRepId;

InvoiceTotal  CustomerName           Country         SalesAgent   
------------  ---------------------  --------------  -------------
3.98          Luís Gonçalves         Brazil          Jane Peacock 
3.96          Luís Gonçalves         Brazil          Jane Peacock 
5.94          Luís Gonçalves         Brazil          Jane Peacock 
0.99          Luís Gonçalves         Brazil          Jane Peacock 
1.98          Luís Gonçalves         Brazil          Jane Peacock 
13.86         Luís Gonçalves         Brazil          Jane Peacock 
8.91          Luís Gonçalves         Brazil          Jane Peacock 
1.98          Leonie Köhler          Germany         Steve Johnson
13.86         Leonie Köhler          Germany         Steve Johnson
8.91          Leonie Köhler          Germany         Steve Johnson
1.98          Leonie Köhler          Germany         Steve Johnson
3.96          Leonie Köhler          Germany         Steve Johnson
5.94          Leonie Köhler          Germany         Steve Joh

### 9. Invoices and total sales in 2009 and 2011

In [17]:
%%script sqlite3 --column --header chinook.db
SELECT strftime('%Y', InvoiceDate) AS Year,
       COUNT(*) AS InvoiceCount,
       ROUND(SUM(Total), 2) AS TotalSales
FROM invoices
WHERE strftime('%Y', InvoiceDate) IN ('2009', '2011')
GROUP BY Year;

Year  InvoiceCount  TotalSales
----  ------------  ----------
2009  83            449.46    
2011  83            469.58    


### 10. Line items on invoice 37

In [18]:
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS LineItemCount
FROM invoice_items
WHERE InvoiceId = 37;

LineItemCount
-------------
4            


### 11. Line items per invoice

In [19]:
%%script sqlite3 --column --header chinook.db
SELECT InvoiceId, COUNT(*) AS LineItemCount
FROM invoice_items
GROUP BY InvoiceId;

InvoiceId  LineItemCount
---------  -------------
1          2            
2          4            
3          6            
4          9            
5          14           
6          1            
7          2            
8          2            
9          4            
10         6            
11         9            
12         14           
13         1            
14         2            
15         2            
16         4            
17         6            
18         9            
19         14           
20         1            
21         2            
22         2            
23         4            
24         6            
25         9            
26         14           
27         1            
28         2            
29         2            
30         4            
31         6            
32         9            
33         14           
34         1            
35         2            
36         2            
37         4            
38         6            


### 12. Track name with each line item

In [20]:
%%script sqlite3 --column --header chinook.db
SELECT ii.*, t.Name AS TrackName
FROM invoice_items AS ii
JOIN tracks AS t ON t.TrackId = ii.TrackId;

InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity  TrackName                                                                                                                  
-------------  ---------  -------  ---------  --------  ---------------------------------------------------------------------------------------------------------------------------
1              1          2        0.99       1         Balls to the Wall                                                                                                          
2              1          4        0.99       1         Restless and Wild                                                                                                          
3              2          6        0.99       1         Put The Finger On You                                                                                                      
4              2          8        0.99       1         Inject The Venom                            

### 13. Track and artist name with each line item

In [21]:
%%script sqlite3 --column --header chinook.db
SELECT ii.*, t.Name AS TrackName, ar.Name AS ArtistName
FROM invoice_items AS ii
JOIN tracks  AS t  ON t.TrackId   = ii.TrackId
JOIN albums  AS al ON al.AlbumId  = t.AlbumId
JOIN artists AS ar ON ar.ArtistId = al.ArtistId;

InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity  TrackName                                                                                                                    ArtistName                                                                           
-------------  ---------  -------  ---------  --------  ---------------------------------------------------------------------------------------------------------------------------  -------------------------------------------------------------------------------------
1              1          2        0.99       1         Balls to the Wall                                                                                                            Accept                                                                               
2              1          4        0.99       1         Restless and Wild                                                                                                            Accept            

### 14. Invoices per country

In [22]:
%%script sqlite3 --column --header chinook.db
SELECT BillingCountry, COUNT(*) AS InvoiceCount
FROM invoices
GROUP BY BillingCountry
ORDER BY InvoiceCount DESC;

BillingCountry  InvoiceCount
--------------  ------------
USA             91          
Canada          56          
France          35          
Brazil          35          
Germany         28          
United Kingdom  21          
Portugal        14          
Czech Republic  14          
India           13          
Sweden          7           
Spain           7           
Poland          7           
Norway          7           
Netherlands     7           
Italy           7           
Ireland         7           
Hungary         7           
Finland         7           
Denmark         7           
Chile           7           
Belgium         7           
Austria         7           
Australia       7           
Argentina       7           


### 15. Tracks per playlist

In [23]:
%%script sqlite3 --column --header chinook.db
SELECT p.Name AS PlaylistName, COUNT(pt.TrackId) AS TrackCount
FROM playlists AS p
LEFT JOIN playlist_track AS pt ON pt.PlaylistId = p.PlaylistId
GROUP BY p.PlaylistId, p.Name
ORDER BY TrackCount DESC;

PlaylistName                TrackCount
--------------------------  ----------
Music                       3290      
Music                       3290      
90’s Music                  1477      
TV Shows                    213       
TV Shows                    213       
Classical                   75        
Brazilian Music             39        
Heavy Metal Classic         26        
Classical 101 - Deep Cuts   25        
Classical 101 - Next Steps  25        
Classical 101 - The Basics  25        
Grunge                      15        
Music Videos                1         
On-The-Go 1                 1         
Movies                      0         
Audiobooks                  0         
Audiobooks                  0         
Movies                      0         


### 16. All tracks with no IDs (album, media type, genre)

In [24]:
%%script sqlite3 --column --header chinook.db
SELECT t.Name AS TrackName, al.Title AS AlbumName,
       mt.Name AS MediaType, g.Name AS Genre
FROM tracks AS t
JOIN albums      AS al ON al.AlbumId     = t.AlbumId
JOIN media_types AS mt ON mt.MediaTypeId = t.MediaTypeId
JOIN genres      AS g  ON g.GenreId      = t.GenreId;

TrackName                                                                                                                    AlbumName                                                                                        MediaType                    Genre             
---------------------------------------------------------------------------------------------------------------------------  -----------------------------------------------------------------------------------------------  ---------------------------  ------------------
For Those About To Rock (We Salute You)                                                                                      For Those About To Rock We Salute You                                                            MPEG audio file              Rock              
Balls to the Wall                                                                                                            Balls to the Wall                                                

### 17. All invoices with number of line items

In [25]:
%%script sqlite3 --column --header chinook.db
SELECT i.*, COUNT(ii.InvoiceLineId) AS LineItemCount
FROM invoices AS i
LEFT JOIN invoice_items AS ii ON ii.InvoiceId = i.InvoiceId
GROUP BY i.InvoiceId;

InvoiceId  CustomerId  InvoiceDate          BillingAddress                            BillingCity          BillingState  BillingCountry  BillingPostalCode  Total  LineItemCount
---------  ----------  -------------------  ----------------------------------------  -------------------  ------------  --------------  -----------------  -----  -------------
1          2           2009-01-01 00:00:00  Theodor-Heuss-Straße 34                   Stuttgart                          Germany         70174              1.98   2            
2          4           2009-01-02 00:00:00  Ullevålsveien 14                          Oslo                               Norway          0171               3.96   4            
3          8           2009-01-03 00:00:00  Grétrystraat 63                           Brussels                           Belgium         1000               5.94   6            
4          14          2009-01-06 00:00:00  8210 111 ST NW                            Edmonton             AB      

### 18. Total sales by sales agent

In [26]:
%%script sqlite3 --column --header chinook.db
SELECT e.FirstName || ' ' || e.LastName AS SalesAgent,
       ROUND(SUM(i.Total), 2) AS TotalSales
FROM employees AS e
JOIN customers AS c ON c.SupportRepId = e.EmployeeId
JOIN invoices  AS i ON i.CustomerId  = c.CustomerId
GROUP BY e.EmployeeId
ORDER BY TotalSales DESC;

SalesAgent     TotalSales
-------------  ----------
Jane Peacock   833.04    
Margaret Park  775.4     
Steve Johnson  720.16    


### 19. Top sales agent in 2009

In [27]:
%%script sqlite3 --column --header chinook.db
WITH agent_sales AS (
    SELECT e.FirstName || ' ' || e.LastName AS SalesAgent, ROUND(SUM(i.Total), 2) AS TotalSales
    FROM employees AS e
    JOIN customers AS c ON c.SupportRepId = e.EmployeeId
    JOIN invoices AS i ON i.CustomerId = c.CustomerId
    WHERE strftime('%Y', i.InvoiceDate) = '2009'
    GROUP BY e.EmployeeId
)
SELECT * FROM agent_sales
WHERE TotalSales = (SELECT MAX(TotalSales) FROM agent_sales);

SalesAgent     TotalSales
-------------  ----------
Steve Johnson  164.34    


### 20. Top sales agent in 2010

In [28]:
%%script sqlite3 --column --header chinook.db
WITH agent_sales AS (
    SELECT e.FirstName || ' ' || e.LastName AS SalesAgent, ROUND(SUM(i.Total), 2) AS TotalSales
    FROM employees AS e
    JOIN customers AS c ON c.SupportRepId = e.EmployeeId
    JOIN invoices AS i ON i.CustomerId = c.CustomerId
    WHERE strftime('%Y', i.InvoiceDate) = '2010'
    GROUP BY e.EmployeeId
)
SELECT * FROM agent_sales
WHERE TotalSales = (SELECT MAX(TotalSales) FROM agent_sales);

SalesAgent    TotalSales
------------  ----------
Jane Peacock  221.92    


### 21. Top sales agent overall

In [29]:
%%script sqlite3 --column --header chinook.db
WITH agent_sales AS (
    SELECT e.FirstName || ' ' || e.LastName AS SalesAgent, ROUND(SUM(i.Total), 2) AS TotalSales
    FROM employees AS e
    JOIN customers AS c ON c.SupportRepId = e.EmployeeId
    JOIN invoices AS i ON i.CustomerId = c.CustomerId
    GROUP BY e.EmployeeId
)
SELECT * FROM agent_sales
WHERE TotalSales = (SELECT MAX(TotalSales) FROM agent_sales);

SalesAgent    TotalSales
------------  ----------
Jane Peacock  833.04    


### 22. Customers per sales agent

In [30]:
%%script sqlite3 --column --header chinook.db
SELECT e.FirstName || ' ' || e.LastName AS SalesAgent,
       COUNT(c.CustomerId) AS CustomerCount
FROM employees AS e
JOIN customers AS c ON c.SupportRepId = e.EmployeeId
GROUP BY e.EmployeeId
ORDER BY CustomerCount DESC;

SalesAgent     CustomerCount
-------------  -------------
Jane Peacock   21           
Margaret Park  20           
Steve Johnson  18           


### 23. Total sales by country

In [31]:
%%script sqlite3 --column --header chinook.db
SELECT BillingCountry, ROUND(SUM(Total), 2) AS TotalSales
FROM invoices
GROUP BY BillingCountry
ORDER BY TotalSales DESC;

BillingCountry  TotalSales
--------------  ----------
USA             523.06    
Canada          303.96    
France          195.1     
Brazil          190.1     
Germany         156.48    
United Kingdom  112.86    
Czech Republic  90.24     
Portugal        77.24     
India           75.26     
Chile           46.62     
Ireland         45.62     
Hungary         45.62     
Austria         42.62     
Finland         41.62     
Netherlands     40.62     
Norway          39.62     
Sweden          38.62     
Spain           37.62     
Poland          37.62     
Italy           37.62     
Denmark         37.62     
Belgium         37.62     
Australia       37.62     
Argentina       37.62     


### 24. Most purchased track of 2013 (note: many tracks tie at 1 sale)

In [32]:
%%script sqlite3 --column --header chinook.db
WITH track_sales AS (
    SELECT t.TrackId, t.Name AS TrackName, SUM(ii.Quantity) AS UnitsSold
    FROM invoice_items AS ii
    JOIN invoices AS i ON i.InvoiceId = ii.InvoiceId
    JOIN tracks   AS t ON t.TrackId   = ii.TrackId
    WHERE strftime('%Y', i.InvoiceDate) = '2013'
    GROUP BY t.TrackId
)
SELECT *
FROM track_sales
WHERE UnitsSold = (SELECT MAX(UnitsSold) FROM track_sales);

TrackId  TrackName                                            UnitsSold
-------  ---------------------------------------------------  ---------
437      Black Diamond                                        1        
443      Shock Me                                             1        
449      Calling Dr. Love                                     1        
455      God Gave Rock 'n' Roll To You                        1        
461      Surrender                                            1        
467      Best Thing                                           1        
473      When I Come Around                                   1        
479      Walking Contradiction                                1        
485      Minority                                             1        
494      Slave                                                1        
503      Começaria Tudo Outra Vez                             1        
512      Comportamento Geral                                  1 

### 25. Top 5 most purchased tracks overall (note: ties at 2 sales)

In [33]:
%%script sqlite3 --column --header chinook.db
SELECT t.TrackId, t.Name AS TrackName, SUM(ii.Quantity) AS UnitsSold
FROM invoice_items AS ii
JOIN tracks AS t ON t.TrackId = ii.TrackId
GROUP BY t.TrackId
ORDER BY UnitsSold DESC, TrackName
LIMIT 5;

TrackId  TrackName                 UnitsSold
-------  ------------------------  ---------
298      A Cor Do Sol              2        
2792     A Melhor Forma            2        
1099     A Novidade (Live)         2        
1785     Abraham, Martin And John  2        
1344     Aces High                 2        


### 26. Top 3 best-selling artists

In [34]:
%%script sqlite3 --column --header chinook.db
SELECT ar.Name AS ArtistName,
       SUM(ii.Quantity) AS UnitsSold,
       ROUND(SUM(ii.UnitPrice * ii.Quantity), 2) AS Revenue
FROM invoice_items AS ii
JOIN tracks  AS t  ON t.TrackId   = ii.TrackId
JOIN albums  AS al ON al.AlbumId  = t.AlbumId
JOIN artists AS ar ON ar.ArtistId = al.ArtistId
GROUP BY ar.ArtistId
ORDER BY Revenue DESC
LIMIT 3;

ArtistName   UnitsSold  Revenue
-----------  ---------  -------
Iron Maiden  140        138.6  
U2           107        105.93 
Metallica    91         90.09  


### 27. Most purchased media type

In [35]:
%%script sqlite3 --column --header chinook.db
SELECT mt.Name AS MediaType, SUM(ii.Quantity) AS UnitsSold
FROM invoice_items AS ii
JOIN tracks      AS t  ON t.TrackId      = ii.TrackId
JOIN media_types AS mt ON mt.MediaTypeId = t.MediaTypeId
GROUP BY mt.MediaTypeId
ORDER BY UnitsSold DESC
LIMIT 1;

MediaType        UnitsSold
---------------  ---------
MPEG audio file  1976     


In [36]:
import re

queries = {}  # a dict removes duplicates if you re-ran a cell
for cell in _ih:  # IPython's record of every cell you ran
    if cell.lstrip().startswith("%%script sqlite3"):
        sql = cell.split("\n", 1)[1].strip()  # drop the %%script line
        if sql and "SELECT" in sql.upper() and not sql.startswith("."):
            queries[sql] = None

with open("chinook-queries.sql", "w") as f:
    f.write("-- chinook-queries.sql\n-- Chinook database (SQLite)\n\n")
    for n, sql in enumerate(queries, 1):
        f.write(f"-- Query {n}\n{sql}\n\n")

print(f"Saved {len(queries)} queries")

from google.colab import files
files.download("chinook-queries.sql")  # downloads it to your computer

Saved 0 queries


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## EXTRA CREDIT:
* Use a CTE
* Use a query that joins two or more tables.
* Make a model to see if you can predict something ( e.g. GNB classifier, linear regression )
* Come up with something else cool to do with your data
